<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####Other Types of joins
1. Natural Join - Automatically create join criteria on the same column names (Applies to Inner and Outer Joins)
2. Cross Join - Join without any join criteria (all possible combinations)
3. Self Join - Join a table with itself (Applies to Inner, Outer, and Cross Joins)
4. Semi Join - Take records from the left side when it matches with the right side (Correlated EXISTS)
5. Anti Join - Take records from the left side when it doesn not match with the right side (Correlated NOT EXISTS)


Q1. Show me a facility bookings report as the following. (Prefer Natural Join)
```
member_id | first_name | last_name | facility_name | slots | booking_amount | start_time
--------------------------------------------------------------------------------------------
```
The report must meet the following criteria.
1. Facility bookings made by a person whose last name is Smith
2. He has booked more than 5 slots in a single booking
3. Report should be sorted by first name of the member in ascending order and booking amount in descending order

In [0]:
from pyspark.sql.functions import col

bookings_df = spark.table("dev.spark_db.bookings").filter("slots > 5")
members_df = spark.table("dev.spark_db.members").filter("last_name == 'Smith'")
facilities_df = spark.table("dev.spark_db.facilities")

bmf_df = bookings_df.join(members_df, ["member_id"], "inner").join(facilities_df, "facility_id")

report_df = (
    bmf_df.selectExpr("member_id", "first_name", "last_name", "facility_name", "slots",
                "slots * member_cost as booking_amount", "start_time")
        .orderBy("first_name", col("booking_amount").desc())
)

display(report_df)

member_id,first_name,last_name,facility_name,slots,booking_amount,start_time
1,Darren,Smith,Badminton Court,6,0.0,2022-08-07T09:00:00.000Z
1,Darren,Smith,Badminton Court,9,0.0,2022-08-28T13:30:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-09-30T14:00:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-07-29T12:00:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-07-27T12:00:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-09-09T13:00:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-09-07T14:00:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-09-10T09:00:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-07-09T09:00:00.000Z
1,Darren,Smith,Badminton Court,6,0.0,2022-08-01T09:30:00.000Z


Q2. Prepare a member bookings report as the following (Prefer Natural Join)
```
booking_id | facility_name | slots | first_name | last_name | address
```
Ensure the following
1. Consider only regular memebrs (not guest) and direct members(not recomended by any other member)
2. Consider only bookings for more than 8 hours
3. Ensure all regular and direct members are listed even if they have no 8 hour bookings
4. Ensure all 8 hour bookings are listed even if they are not made by regular and direct members
5. Sort the report by slots and first name in ascending order

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id != 0  and recommended_by is null")
bookings_df = spark.table("dev.spark_db.bookings").filter("slots > 8")
facilities_df = spark.table("dev.spark_db.facilities")

joined_df = (
    members_df.join(bookings_df, "member_id", "full")
            .join(facilities_df, "facility_id", "left")
)

report_df = (
    joined_df.select("booking_id", "facility_name", "slots", "first_name", "last_name", "address")
        .orderBy("slots", "first_name")
)

display(report_df)

booking_id,facility_name,slots,first_name,last_name,address
null,null,null,Burton,Tracy,"3 Tunisia Drive, Boston"
null,null,null,Darren,Smith,"3 Funktown, Denzington, Boston"
null,null,null,David,Farrell,"437 Granite Farm Road, Westford"
null,null,null,Hyacinth,Tupperware,"33 Cheerful Plaza, Drake Road, Westford"
null,null,null,Jemima,Farrell,"103 Firth Avenue, North Reading"
null,null,null,Tim,Rownam,"23 Highway Way, Boston"
null,null,null,Tracy,Smith,"8 Bloomsbury Close, New York"
530,Tennis Court 1,9,null,null,null
1757,Tennis Court 2,9,null,null,null
3563,Tennis Court 1,9,null,null,null


Q3. How many bookings are possible when each member is booking a facility exactly once in a month?\
Show all possible combinations

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id > 0")
facilities_df = spark.table("dev.spark_db.facilities")

report_df = (
    members_df.crossJoin(facilities_df)
        .select("first_name", "last_name", "facility_name")
)

report_df.display()

first_name,last_name,facility_name
Darren,Smith,Tennis Court 2
Tracy,Smith,Badminton Court
Tim,Rownam,Table Tennis
Janice,Joplette,Massage Room 1
Gerald,Butters,Massage Room 2
Burton,Tracy,Squash Court
Nancy,Dare,Snooker Table
Tim,Boothe,Pool Table
Ponder,Stibbons,Tennis Court 1
Charles,Owen,Tennis Court 2


Q4. Prepare a report for members and who recomended them as the following
```
member_id | Member Name | Recommended By
--------------------------------------------
```

In [0]:
from pyspark.sql.functions import expr, concat_ws

members_df = spark.table("dev.spark_db.members")

report_df = (
    members_df.alias("m")
        .join(members_df.alias("r"), expr("m.recommended_by==r.member_id"), "inner")
        .select("m.member_id",
                concat_ws(" ", "m.first_name", "m.last_name").alias("Member Name"),
                concat_ws(" ", "r.first_name", "r.last_name").alias("Recommended By"))
)

report_df.display()

member_id,Member Name,Recommended By
21,Anna Mackenzie,Darren Smith
36,Erica Crumpet,Tracy Smith
8,Tim Boothe,Tim Rownam
11,David Jones,Janice Joplette
20,Matthew Genting,Gerald Butters
9,Ponder Stibbons,Burton Tracy
15,Florence Bader,Ponder Stibbons
26,Douglas Jones,David Jones
17,David Pinker,Jemima Farrell
24,Ramnaresh Sarwin,Florence Bader


Q5. Prepare a list of members who made at least one booking. (Use SEMI Join)
```
member_id | first_name | last_name | address
-----------------------------------------------
```

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id > 0").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")

report_df = (
    members_df.join(bookings_df, expr("m.member_id == b.member_id"), "left_semi")
        .select("member_id", "first_name", "last_name", "address")
)

report_df.display()

member_id,first_name,last_name,address
1,Darren,Smith,"8 Bloomsbury Close, Boston"
2,Tracy,Smith,"8 Bloomsbury Close, New York"
3,Tim,Rownam,"23 Highway Way, Boston"
4,Janice,Joplette,"20 Crossing Road, New York"
5,Gerald,Butters,"1065 Huntingdon Avenue, Boston"
6,Burton,Tracy,"3 Tunisia Drive, Boston"
7,Nancy,Dare,"6 Hunting Lodge Way, Boston"
8,Tim,Boothe,"3 Bloomsbury Close, Reading, 00234"
9,Ponder,Stibbons,"5 Dragons Way, Winchester"
10,Charles,Owen,"52 Cheshire Grove, Winchester, 28563"


Q6. Prepare a list of members who never made any bookings. (Use ANTI Join)
```
member_id | first_name | last_name | address
-----------------------------------------------
```

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id > 0").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")

report_df = (
    members_df.join(bookings_df, expr("m.member_id == b.member_id"), "left_anti")
        .select("member_id", "first_name", "last_name", "address")
)

report_df.display()

member_id,first_name,last_name,address
37,Darren,Smith,"3 Funktown, Denzington, Boston"


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>